## Run test: HuggingFaceTB/SmolVLM-256M-Instruct

Model page: https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct

Task: `image-text-to-text`  ·  Library: `transformers`  ·  Source: official-template

> Reproduced from HuggingFace's official auto-generated colab/transformers snippet (identical code). Local inference on 8×W7900 with `device_map="auto"` added.

In [ ]:
!pip install -U transformers accelerate

In [ ]:
from transformers import AutoConfig

model_path = "HuggingFaceTB/SmolVLM-256M-Instruct"
cfg = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
print("config:", type(cfg).__name__, "model_type=", getattr(cfg, "model_type", None))


In [ ]:
# Load model directly (fixed: VLM uses image-text-to-text head)
from transformers import AutoProcessor, AutoModelForImageTextToText

model_path = "HuggingFaceTB/SmolVLM-256M-Instruct"
proc = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(model_path, torch_dtype="auto", device_map="auto", trust_remote_code=True)
print("processor:", type(proc).__name__)
print("model:", type(model).__name__)


In [ ]:
# HF Colab aligned image prompt and minimal generation.
messages = [{'role': 'user',
  'content': [{'type': 'image',
               'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
              {'type': 'text', 'text': 'What animal is on the candy?'}]}]
inputs = proc.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(proc.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
